# Multi Document Summurizer

In [ ]:
# !pip install summa pypdf torch transformers

## Part 1: Extractive Summarization
Use summa to extract top sentences from each document separately.

In [ ]:
from summa import summarizer
from pypdf import PdfReader

In [ ]:
# files = input("Enter the file names (space separated): ").split(" ")
files = ["one.pdf", "two.pdf", "three.pdf"]
print("Files entered:", files)

In [ ]:
def summarize_extract(file):
    reader = PdfReader(file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    summary = summarizer.summarize(text, ratio=0.3)
    return summary

In [ ]:
extractive_summaries = []
for file in files:
    summary = summarize_extract(file)
    print(f"Summary for {file}:\n{summary}\n")
    extractive_summaries.append(summary)

In [ ]:
len(extractive_summaries[0])

## Part 2: Abstraction summarization
In this phase the summaries extracted form each document are combined and summarised together using distilBart-12-6 model.

In [ ]:
from transformers import (
    AutoTokenizer,
    BartForConditionalGeneration,
    AutoModelForSeq2SeqLM,
)

In [ ]:
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

In [ ]:
finalInput = ""
for extSum in extractive_summaries:
    inputs = tokenizer(extSum, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, min_length=512, max_length=720, early_stopping=True)
    abstractive_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    finalInput += abstractive_summary + " [SEP] "
print(f"Final Abstractive Summary:\n{finalInput}\n")

In [ ]:
model_name_large = "facebook/bart-large-cnn"
tokenizer_large = AutoTokenizer.from_pretrained(model_name_large)
model_large = AutoModelForSeq2SeqLM.from_pretrained(model_name_large)

In [ ]:
inputs = tokenizer_large(finalInput, return_tensors="pt", max_length=1024)
summary_ids = model_large.generate(inputs["input_ids"], num_beams=4, min_length=512, max_length=4096, early_stopping=True)
finalSummary = tokenizer_large.decode(summary_ids[0], skip_special_tokens=True)

print(f"Final Summary:\n{finalSummary}\n")